In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/processed/complaints_clean.parquet")
print(df.shape)

(99709, 21)


In [2]:
from statsmodels.stats.proportion import proportions_ztest

closed = df[df["company_response_to_consumer"].str.startswith("Closed")].copy()
closed["got_relief"] = closed["company_response_to_consumer"].isin(
    ["Closed with monetary relief", "Closed with non-monetary relief"]
)

g = closed.groupby("is_credit_reporting")["got_relief"].agg(["sum", "count"])
g["rate"] = g["sum"] / g["count"]
print(g)

z, p = proportions_ztest(g["sum"].values, g["count"].values)
diff = g.loc[True, "rate"] - g.loc[False, "rate"]
print(f"\nDifference (credit reporting minus others): {diff:.1%}")
print(f"p-value: {p:.4g}")

                       sum  count      rate
is_credit_reporting                        
False                 2369  10899  0.217359
True                 32749  84040  0.389683

Difference (credit reporting minus others): 17.2%
p-value: 2.737e-269


Credit reporting complaints get relief 38% of the time versus 21% for other products

In [5]:
#Do the three credit bureaus differ in untimely responses?
from scipy.stats import chi2_contingency

bureaus = df[df["company"].str.contains("EQUIFAX|EXPERIAN|TRANSUNION", na=False)].copy()

bureaus["bureau"] = np.select(
    [bureaus["company"].str.contains("EQUIFAX"),
     bureaus["company"].str.contains("EXPERIAN"),
     bureaus["company"].str.contains("TRANSUNION")],
    ["Equifax", "Experian", "TransUnion"],
    default="Other"
)

print(bureaus["bureau"].value_counts())

table = pd.crosstab(bureaus["bureau"], bureaus["timely_response"])
print(table)

chi2, p, dof, expected = chi2_contingency(table)
n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))
print(f"\np-value: {p:.4g}")
print(f"Effect size (Cramer's V): {cramers_v:.3f}")
print("Smallest expected count:", expected.min().round(1))

bureau
TransUnion    31123
Equifax       28922
Experian      26646
Name: count, dtype: int64
timely_response  No    Yes
bureau                    
Equifax           0  28922
Experian          1  26645
TransUnion        0  31123

p-value: 0.3241
Effect size (Cramer's V): 0.005
Smallest expected count: 0.3


In [6]:
#Find unusual months (anamoly detection)
monthly = (df[(df["month"] >= "2024-01") & (df["month"] <= "2026-10")]
           .groupby("month").size().rename("complaints").reset_index())
monthly["date"] = pd.to_datetime(monthly["month"] + "-01")
s = monthly.set_index("date")["complaints"].asfreq("MS")
print(len(s), "months")
s.tail()

34 months


date
2026-06-01    4562
2026-07-01    4867
2026-08-01    4272
2026-09-01    4790
2026-10-01    5099
Freq: MS, Name: complaints, dtype: int64

In [7]:
#Build the monthly series
monthly = (df[(df["month"] >= "2024-01") & (df["month"] <= "2026-10")]
           .groupby("month").size().rename("complaints").reset_index())
monthly["date"] = pd.to_datetime(monthly["month"] + "-01")
s = monthly.set_index("date")["complaints"].asfreq("MS")
print(len(s), "months")
s.tail()

34 months


date
2026-06-01    4562
2026-07-01    4867
2026-08-01    4272
2026-09-01    4790
2026-10-01    5099
Freq: MS, Name: complaints, dtype: int64

In [8]:
#Decompose and flag
from statsmodels.tsa.seasonal import STL

res = STL(s, period=12, robust=True).fit()
resid = res.resid
mad = np.median(np.abs(resid - np.median(resid)))
monthly["robust_z"] = 0.6745 * (resid.values - np.median(resid)) / mad
monthly["is_anomaly"] = monthly["robust_z"].abs() > 3
monthly["trend"] = res.trend.values

print(monthly[monthly["is_anomaly"]][["month", "complaints", "robust_z"]])

      month  complaints      robust_z
12  2025-01        3184  5.437915e+14
13  2025-02        1988 -2.626242e+14
14  2025-03        2428 -6.804402e+14
15  2025-04        2573 -2.874269e+14
16  2025-05        2600 -1.059661e+14
17  2025-06        2643 -4.327671e+13
18  2025-07        3168  2.104662e+14
19  2025-08        3748  7.509916e+14
20  2025-09        3741  5.548746e+14
21  2025-10        3960  4.345350e+14
22  2025-11        3838  3.709750e+00


In [9]:
#Forecast on last 6 months
from statsmodels.tsa.holtwinters import ExponentialSmoothing

train, test = s.iloc[:-6], s.iloc[-6:]
model = ExponentialSmoothing(train, trend="add", damped_trend=True).fit()
pred = model.forecast(6)

mae_model = (test - pred).abs().mean()
naive = pd.Series(train.iloc[-1], index=test.index)  # "next months = last month"
mae_naive = (test - naive).abs().mean()

print(f"Model average error: {mae_model:.0f} complaints/month")
print(f"Naive average error: {mae_naive:.0f} complaints/month")

Model average error: 1295 complaints/month
Naive average error: 436 complaints/month


In [10]:
#Forecast Nov 2026 to Apr 2027
final = ExponentialSmoothing(s, trend="add", damped_trend=True).fit()
fc = final.forecast(6)
err = np.std(final.resid)

out = pd.concat([
    pd.DataFrame({"date": s.index, "value": s.values, "type": "Actual"}),
    pd.DataFrame({"date": fc.index, "value": fc.values, "type": "Forecast",
                  "lower": fc.values - 1.96 * err, "upper": fc.values + 1.96 * err})
])
out.tail(8)

,date,value,type,lower,upper
32,2026-09-01,4790.000000,Actual,NaN,NaN
33,2026-10-01,5099.000000,Actual,NaN,NaN
0,2026-11-01,5165.980921,Forecast,4290.886159,6041.075683
1,2026-12-01,5285.387581,Forecast,4410.292819,6160.482343
2,2027-01-01,5404.037166,Forecast,4528.942404,6279.131929
3,2027-02-01,5521.934477,Forecast,4646.839715,6397.029239
4,2027-03-01,5639.084283,Forecast,4763.989520,6514.179045
5,2027-04-01,5755.491322,Forecast,4880.396560,6630.586085


In [11]:
monthly[["date", "complaints", "trend", "robust_z", "is_anomaly"]].to_csv(
    "../data/processed/anomalies.csv", index=False)
out.to_csv("../data/processed/forecast.csv", index=False)
print("Saved")

Saved
